# ΔP as a gate on motifbreakR: Malinois and Kircher

motifbreakR says how much a variant changes a motif and in which direction (alleleDiff). The CAV says whether, in
this 600 bp context, the change actually moves binding (ΔP = P(bound | alt) − P(bound | ref)). Here ΔP does not
predict on its own and nothing is fitted: it **gates** motifbreakR's prediction.

For every motifbreakR call (variant × TF), ΔP is that TF's ΔP at the variant (the TF's occurrence with the largest
|ΔP| among those containing the variant). Gate = a number between 0 and 1: the **percentile of the call within its TF**
(over all calls of that TF), so the TFs' different ΔP scales do not enter. Gated score = alleleDiff × gate; per
variant the call with the largest |gated score| is kept, exactly like motifbreakR keeps the largest |alleleDiff|.
Variants without a call stay 0, so both rules cover the same variants. Continuous gates only, fixed in advance:

| rule | gate | idea |
|---|---|---|
| **ΔP gate** (main) | percentile of \|ΔP\| | trust the PWM change as much as the CAV says binding moves |
| ΔP gate, 0.5 + percentile | 0.5 + percentile of \|ΔP\| (0.5–1.5) | the same, but never close to 0 |
| ΔP gate, direction | percentile of sign(alleleDiff) × ΔP | high when ΔP moves the same way as the PWM (both lose, or both gain) |
| P gate (reference) | percentile of P(bound \| ref) | the earlier idea: trust the PWM change at sites the CAV calls bound |

Scores: Pearson with the measured effect and AUROC for clear effects, against motifbreakR on the same variants.
Null N2: ΔP (and P) shuffled among the calls of the same TF, the whole rule redone. TF tiers by CAV AUC.
CPU only; reads the ΔP already computed by `Malinois MPRA 5` and `Kircher ΔP`. About 10–20 minutes.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 0. Shared functions

In [2]:
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, json, re, time
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

MAL = f"{WORK_DIR}/malinois"
POOL = f"{WORK_DIR}/fcav_pool"
OUT = f"{WORK_DIR}/deltaP_gate"
os.makedirs(OUT, exist_ok=True)
LAYER, N_PERM = 24, 200
RULES = ["ΔP gate", "ΔP gate, 0.5 + percentile", "ΔP gate, direction", "P gate (reference)"]
pd.set_option("display.width", 250)
fmt = lambda x: f"{x:+.3f}"
AUC = {}
for f in os.listdir(f"{POOL}/fits/nt500m"):
    if f.endswith(".npz") and not f.endswith(".part.npz"):
        z = np.load(f"{POOL}/fits/nt500m/{f}", allow_pickle=False)
        AUC[f[:-4]] = float(np.asarray(z["auc"])[[int(l) for l in z["layers"]].index(LAYER)])


def pear(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if a.std() == 0 or b.std() == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def auroc(lab, s):
    lab = np.asarray(lab, bool)
    return roc_auc_score(lab, s) if 0 < lab.sum() < len(lab) else np.nan


def tf_pct(x, g):
    # percentile (0..1) of x among the entries with the same g
    o = np.lexsort((x, g))
    gs = g[o]
    st = np.r_[0, np.flatnonzero(gs[1:] != gs[:-1]) + 1]
    cnt = np.diff(np.r_[st, len(gs)])
    q = np.empty(len(x)); q[o] = (np.arange(len(gs)) - np.repeat(st, cnt) + 0.5) / np.repeat(cnt, cnt)
    return q


def pick(v, grp, n):
    # per group, the value with the largest |v| (0 for a group without entries)
    out = np.zeros(n)
    if len(v):
        o = np.lexsort((-np.abs(v), grp))
        f = o[np.r_[True, grp[o][1:] != grp[o][:-1]]]
        out[grp[f]] = v[f]
    return out


def rule_scores(T, dP, pref, n):
    # T: one row per motifbreakR call -- ad (alleleDiff), tf (code), grp (variant index); dP, pref aligned with T
    ad, tf, g = T["ad"], T["tf"], T["grp"]
    q = tf_pct(np.abs(dP), tf)
    gates = {"ΔP gate": q, "ΔP gate, 0.5 + percentile": 0.5 + q,
             "ΔP gate, direction": tf_pct(np.sign(ad) * dP, tf), "P gate (reference)": tf_pct(pref, tf)}
    out = {"motifbreakR": pick(ad, g, n)}
    for r, w in gates.items():
        out[r] = pick(ad * w, g, n)
    return out


def shuffle_tf(T, rng):
    # one permutation of the calls within each TF, applied to ΔP and P together
    o = np.lexsort((rng.random(len(T["tf"])), T["tf"]))
    base = np.argsort(T["tf"], kind="stable")
    dP, pref = np.empty(len(o)), np.empty(len(o))
    dP[base], pref[base] = T["dP"][o], T["pref"][o]
    return dP, pref

---
# Section 1 — Malinois

## 1.1 Load: every motifbreakR call with its TF's ΔP and P(bound | ref) at the variant

In [3]:
CELLS3 = ["HepG2", "K562", "SKNSH"]
TIERS = [("all TFs", 0.0), ("AUC >= 0.70", 0.70), ("AUC >= 0.75", 0.75), ("AUC >= 0.80", 0.80)]
ACTIVE_MIN, Z_CLEAR = 1.0, 2.0
V = pd.read_csv(f"{MAL}/variants.csv.gz", index_col="vi", dtype={"chrom": str}, low_memory=False)
rd = lambda d: pd.concat([pd.read_csv(f"{d}/{f}") for f in os.listdir(d) if f.endswith(".csv") and not f.endswith(".part.csv")])
H = pd.read_csv(f"{MAL}/hits.csv")
H = H[~H["leak"]].drop_duplicates(["vi", "uid"])[["vi", "uid", "motif"]].merge(rd(f"{MAL}/palt_nt500m"), on=["vi", "uid"])
H["P_ref"] = rd(f"{MAL}/p_nt500m").set_index("uid")["P"].reindex(H["uid"]).values
H["dP"] = 1 / (1 + np.exp(-H["dec_alt"])) - H["P_ref"]
BEST = (H.assign(a=H["dP"].abs()).sort_values("a", ascending=False, kind="stable").drop_duplicates(["vi", "motif"])
        [["vi", "motif", "dP"]].merge(H.groupby(["vi", "motif"])["P_ref"].max().reset_index(), on=["vi", "motif"]))
CM = pd.read_csv(f"{MAL}/calls.csv").merge(BEST, on=["vi", "motif"])
CM["auc"] = CM["motif"].map(AUC)
print(f"{len(CM):,} motifbreakR calls with a ΔP on {CM['vi'].nunique():,} variants, {CM['motif'].nunique()} TFs")

115,254 motifbreakR calls with a ΔP on 67,447 variants, 251 TFs


## 1.2 Every tier

In [4]:
def mal_table(L):
    L = L.sort_values(["vi", "motif"]).reset_index(drop=True)
    VI = np.unique(L["vi"].values)
    T = dict(ad=L["alleleDiff"].values, tf=pd.factorize(L["motif"])[0], grp=np.searchsorted(VI, L["vi"].values),
             dP=L["dP"].values, pref=L["P_ref"].values)
    Y = {c: V.loc[VI, f"d_{c}"].values for c in CELLS3}
    CL = {c: np.abs(Y[c]) / V.loc[VI, f"se_{c}"].values >= Z_CLEAR for c in CELLS3}
    SETS = {(c, s): np.isfinite(Y[c]) & ((V.loc[VI, f"act_{c}"].values >= ACTIVE_MIN) if s == "active" else True)
            for c in CELLS3 for s in ("all", "active")}
    return T, VI, Y, CL, SETS


def mal_stats(T, VI, Y, SETS, dP, pref):
    S = rule_scores(T, dP, pref, len(VI))
    return {key: np.array([pear(S[r][m], Y[key[0]][m]) for r in ["motifbreakR"] + RULES]) for key, m in SETS.items()}


ROWS = []
t0 = time.time()
for name, thr in TIERS:
    L = CM[CM["auc"] >= thr]
    T, VI, Y, CL, SETS = mal_table(L)
    rng = np.random.default_rng(7)
    obs = mal_stats(T, VI, Y, SETS, T["dP"], T["pref"])
    null = [mal_stats(T, VI, Y, SETS, *shuffle_tf(T, rng)) for _ in range(N_PERM)]
    S = rule_scores(T, T["dP"], T["pref"], len(VI))
    for key, m in SETS.items():
        c, s = key
        o = obs[key]
        nl = np.array([n_[key] for n_ in null])
        row = {"tier": name, "cell": c, "variants": s, "TFs": int(L["motif"].nunique()), "n": int(m.sum()),
               "motifbreakR": o[0], "AUROC motifbreakR": auroc(CL[c][m], np.abs(S["motifbreakR"][m]))}
        for i, r in enumerate(RULES, 1):
            row[f"{r}: gain"] = o[i] - o[0]
            row[f"{r}: p N2"] = np.mean(nl[:, i] >= o[i])
            row[f"{r}: AUROC gain"] = auroc(CL[c][m], np.abs(S[r][m])) - row["AUROC motifbreakR"]
        ROWS.append(row)
    print(f"{name}: {L['motif'].nunique()} TFs, {len(VI):,} variants  [{(time.time() - t0) / 60:.1f} min]", flush=True)
RM = pd.DataFrame(ROWS).set_index(["tier", "cell", "variants"])
RM.to_csv(f"{OUT}/malinois.csv")
print("\nPearson with Δ (motifbreakR) and each gate's gain over it on the same variants; p N2 = share of shuffles of ΔP within TF with at least the gated Pearson\n")
print(RM[["TFs", "n", "motifbreakR"] + [f"{r}: {k}" for r in RULES for k in ("gain", "p N2")]].to_string(float_format=fmt))
print("\nAUROC for clear effects (|Δ| >= 2 SE): motifbreakR and each gate's gain\n")
print(RM[["AUROC motifbreakR"] + [f"{r}: AUROC gain" for r in RULES]].to_string(float_format=fmt))

all TFs: 251 TFs, 67,447 variants  [1.4 min]
AUC >= 0.70: 143 TFs, 48,086 variants  [2.1 min]
AUC >= 0.75: 95 TFs, 35,708 variants  [2.6 min]
AUC >= 0.80: 50 TFs, 22,483 variants  [2.9 min]

Pearson with Δ (motifbreakR) and each gate's gain over it on the same variants; p N2 = share of shuffles of ΔP within TF with at least the gated Pearson

                            TFs      n  motifbreakR  ΔP gate: gain  ΔP gate: p N2  ΔP gate, 0.5 + percentile: gain  ΔP gate, 0.5 + percentile: p N2  ΔP gate, direction: gain  ΔP gate, direction: p N2  P gate (reference): gain  P gate (reference): p N2
tier        cell  variants                                                                                                                                                                                                                                 
all TFs     HepG2 all       251  67447       +0.069         +0.011         +0.005                           +0.011                           +0.005    

---
# Section 2 — Kircher (21 regions)

The motifbreakR calls of `Pooled Regions 2` (code copied from the diagnosis notebook; its region 0 / 8 / 20 Pearson
0.177 / 0.279 / 0.464 must come back), on the SNVs inside at least one occurrence (0 where there is no call), and
the ΔP of `Kircher ΔP`. The within-TF percentile is taken over the calls of all regions together.

In [5]:
HOME = {"HepG2", "K562"}
K_TIERS = [("all TFs", 0.0), ("AUC >= 0.75", 0.75)]
MBR_THRESH, MBR_NEUT = 0.85, 0.4
C = json.load(open(f"{WORK_DIR}/config.json"))
MWJ = json.load(open(f"{POOL}/mpra_windows.json"))
INFO = {int(k): v for k, v in MWJ["regions"].items()}
def _read_meme(path):
    out, name, rows_, ns = {}, None, [], 20
    for line in open(path):
        t = line.split()
        if line.startswith("MOTIF"):
            if name is not None and name not in out:
                out[name] = (np.array(rows_), ns)
            name, rows_, ns = (t[2] if len(t) > 2 else t[1]), [], 20
        elif line.startswith("letter-probability"):
            mm = re.search(r"nsites=\s*([0-9.]+)", line)
            ns = float(mm.group(1)) if mm else 20
        elif name is not None and len(t) == 4 and all(re.fullmatch(r"[0-9.eE+-]+", x) for x in t):
            rows_.append([float(x) for x in t])
    if name is not None and name not in out:
        out[name] = (np.array(rows_), ns)
    return out


MBR_MATS = {}
for name, (f, ns) in _read_meme(C["meme_path"]).items():
    f = f / f.sum(1, keepdims=True)
    pp = (f * ns + 0.25) / (ns + 1)
    S = pp * (pp.max(1) - pp.min(1))[:, None]
    S = np.vstack([S.T, np.zeros(len(S))]).T
    MBR_MATS[name] = (S, S[::-1][:, [3, 2, 1, 0, 4]], S[:, :4].min(1).sum(), S[:, :4].max(1).sum())
ENC = {"A": 0, "C": 1, "G": 2, "T": 3}
WIN = {}
for h in MWJ["hits"]:
    WIN.setdefault(h["region"], []).append(h)


def build(r, HS):
    # verbatim from Pooled Regions 2: SNV x hit rows and motifbreakR calls on the hits HS of region r
    S_ = f"{POOL}/regions/r{r}"
    hits = pd.read_csv(f"{S_}/hits.csv").set_index("hit_id")
    snv = pd.read_csv(f"{S_}/snvs.csv")
    q = pd.read_csv(f"{S_}/snv_hit_dpwm.csv")
    q = q[q["hit_id"].isin(HS)].sort_values(["snv", "hit_id"]).reset_index(drop=True)
    sn = q["snv"].values
    st = np.r_[0, np.flatnonzero(sn[1:] != sn[:-1]) + 1]
    hi = np.searchsorted(HS, q["hit_id"].values)
    n_snv = len(st)
    MOT = hits.loc[HS, "motif"].values
    base = {}
    for h in WIN[r]:
        for i, b in enumerate(h["window"]):
            base.setdefault(h["start"] - C["motif_offset"] + i, b)
    lo = min(base)
    seq = np.array([ENC.get(base.get(x, "N"), 4) for x in range(lo, max(base) + 1)])
    s_ = snv.loc[sn[st]]
    pos, ref, alt = s_["local"].values - lo, s_["ref"].map(ENC).values, s_["alt"].map(ENC).values
    assert (seq[pos] == ref).all(), "rebuilt sequence disagrees with the SNV reference bases"
    out = []
    for m in sorted(set(MOT)):
        if m not in MBR_MATS:
            continue
        S, Sr, smin, smax = MBR_MATS[m]
        w = len(S)
        n = len(seq) - w + 1
        idx = np.arange(n)[:, None] + np.arange(w)[None, :]
        f_fwd, f_rev = S[np.arange(w), seq[idx]].sum(1), Sr[np.arange(w), seq[idx]].sum(1)
        stw = pos[:, None] - np.arange(w)[None, :]
        k = np.broadcast_to(np.arange(w), stw.shape)
        ok = (stw >= 0) & (stw < n)
        stc = np.clip(stw, 0, n - 1)
        ref_f, ref_r = np.where(ok, f_fwd[stc], -np.inf), np.where(ok, f_rev[stc], -np.inf)
        alt_f = ref_f + np.where(ok, S[k, alt[:, None]] - S[k, ref[:, None]], 0)
        alt_r = ref_r + np.where(ok, Sr[k, alt[:, None]] - Sr[k, ref[:, None]], 0)
        sr, sa = np.maximum(ref_f, ref_r).max(1), np.maximum(alt_f, alt_r).max(1)
        pr, pa = (sr - smin) / (smax - smin), (sa - smin) / (smax - smin)
        diff = sa - sr
        keep = np.isfinite(sr) & ((pr > MBR_THRESH) | (pa > MBR_THRESH)) & (np.abs(diff) >= MBR_NEUT)
        out += [(i, m, diff[i]) for i in np.flatnonzero(keep)]
    t = pd.DataFrame(out, columns=["i", "motif", "alleleDiff"])
    t["j"] = np.arange(len(t))
    rows_ = pd.DataFrame({"i": np.repeat(np.arange(n_snv), np.diff(np.r_[st, len(sn)])), "motif": MOT[hi], "h": hi})
    link = t[["j", "i", "motif"]].merge(rows_, on=["i", "motif"])
    keep = np.sort(link["j"].unique())
    t = t.loc[keep].reset_index(drop=True)
    return dict(snv=snv, n_snv=n_snv, snv_id=sn[st], y=snv.loc[sn[st], "log2FC"].values,
                DIFF=t["alleleDiff"].values, TI=t["i"].values, MOTP=t["motif"].values)


KP = pd.read_csv(f"{WORK_DIR}/kircher_deltaP/pairs.csv")
KB = (KP.assign(a=KP["dP"].abs()).sort_values("a", ascending=False, kind="stable").drop_duplicates(["region", "snv", "motif"])
      [["region", "snv", "motif", "dP"]].merge(KP.groupby(["region", "snv", "motif"])["P_ref"].max().reset_index(),
                                               on=["region", "snv", "motif"]))
HW = [h for h in MWJ["hits"] if h["motif"] in AUC]


def kircher_tier(thr):
    regs, calls = {}, []
    for r in sorted(INFO):
        hs = np.array(sorted(h["hit_id"] for h in HW if h["region"] == r and AUC[h["motif"]] >= thr))
        if not len(hs):
            continue
        R = build(r, hs)
        if R["n_snv"] < 20:
            continue
        regs[r] = R
        calls.append(pd.DataFrame({"region": r, "i": R["TI"], "snv": R["snv_id"][R["TI"]], "motif": R["MOTP"], "ad": R["DIFF"]}))
    Cc = pd.concat(calls, ignore_index=True).merge(KB, on=["region", "snv", "motif"], how="left")
    miss = Cc["dP"].isna().mean()
    Cc = Cc[Cc["dP"].notna()].reset_index(drop=True)
    off = {r: o for r, o in zip(regs, np.cumsum([0] + [regs[r]["n_snv"] for r in regs])[:-1])}
    T = dict(ad=Cc["ad"].values, tf=pd.factorize(Cc["motif"])[0], grp=(Cc["region"].map(off) + Cc["i"]).values,
             dP=Cc["dP"].values, pref=Cc["P_ref"].values)
    return regs, off, T, miss


def kircher_stats(regs, off, T, dP, pref):
    n = sum(R["n_snv"] for R in regs.values())
    S = rule_scores(T, dP, pref, n)
    return {r: np.array([pear(S[k][off[r]:off[r] + R["n_snv"]], R["y"]) for k in ["motifbreakR"] + RULES]) for r, R in regs.items()}


KROWS, KREG = [], []
for name, thr in K_TIERS:
    regs, off, T, miss = kircher_tier(thr)
    obs = kircher_stats(regs, off, T, T["dP"], T["pref"])
    if name == "all TFs":
        print(f"calls without a ΔP (dropped): {miss:.1%}")
        print("check (motifbreakR, Pooled Regions 2): " + ", ".join(f"region {r} {obs[r][0]:.3f} (was {w})"
                                                                     for r, w in ((0, 0.177), (8, 0.279), (20, 0.464)) if r in obs))
    rng = np.random.default_rng(7)
    null = [kircher_stats(regs, off, T, *shuffle_tf(T, rng)) for _ in range(N_PERM)]
    for r, o in obs.items():
        KREG.append({"tier": name, "region": r, "cell": INFO[r]["cell"], **dict(zip(["motifbreakR"] + RULES, o))})
    for h in (True, False):
        rs = [r for r in regs if (set(INFO[r]["cell"].split(",")) <= HOME) == h]
        O = np.array([obs[r] for r in rs])
        NL = np.array([[n_[r] for r in rs] for n_ in null])          # (perm, region, rule)
        row = {"tier": name, "regions": f"{'HepG2/K562' if h else 'other cells'} ({len(rs)})", "motifbreakR": np.nanmean(O[:, 0])}
        for i, r in enumerate(RULES, 1):
            row[f"{r}: gain"] = np.nanmean(O[:, i]) - np.nanmean(O[:, 0])
            row[f"{r}: wins"] = f"{int((O[:, i] > O[:, 0]).sum())}/{len(rs)}"
            row[f"{r}: p N2"] = np.mean(np.nanmean(NL[:, :, i], 1) >= np.nanmean(O[:, i]))
        KROWS.append(row)
KR = pd.DataFrame(KREG).set_index(["tier", "region"])
KS = pd.DataFrame(KROWS).set_index(["tier", "regions"])
KR.to_csv(f"{OUT}/kircher_regions.csv"); KS.to_csv(f"{OUT}/kircher_summary.csv")
print("\nPearson with the measured log2FC per region\n")
print(KR.to_string(float_format=fmt))
print("\nmean over regions: motifbreakR, and each gate's gain, regions where it wins, p N2\n")
print(KS.to_string(float_format=fmt))

calls without a ΔP (dropped): 0.0%
check (motifbreakR, Pooled Regions 2): region 0 0.177 (was 0.177), region 8 0.279 (was 0.279), region 20 0.464 (was 0.464)

Pearson with the measured log2FC per region

                              cell  motifbreakR  ΔP gate  ΔP gate, 0.5 + percentile  ΔP gate, direction  P gate (reference)
tier        region                                                                                                         
all TFs     0                HepG2       +0.177   +0.225                     +0.208              +0.229              +0.278
            1                 K562       +0.259   +0.305                     +0.295              +0.293              +0.309
            2                HaCaT       -0.003   +0.012                     +0.005              +0.022              +0.051
            3             Neuro-2a       +0.102   +0.109                     +0.112              +0.087              +0.083
            4              HEK293T       +0.075   +0

## End

Outputs in `deltaP_gate/`: `malinois.csv`, `kircher_regions.csv`, `kircher_summary.csv`.